# PyGLDA v2 — regional GRACE data assimilation, step by step

**Developer:** Fan Yang (fany@plan.aau.dk), Geodesy Group, Aalborg University
**Reference:** Yang et al., *Geosci. Model Dev.*, 18, 6195–6217, 2025 — https://doi.org/10.5194/gmd-18-6195-2025

This notebook walks through one complete regional experiment of PyGLDA v2: assimilating monthly
GRACE terrestrial water storage (TWS) into the WaterGAP 2.2e hydrological model with an ensemble
Kalman filter, for the **Danube** basin (case `demo_2`). Each step explains what happens, runs it,
and looks at what it produced before moving on.

```
 configuration ─► basin mask ─► perturbation ─► GRACE observations
                                                        │
      spin-up (MPI) ─► open loop (MPI) ─► collect OL ──►│──► DA (MPI) ─► collect DA
                                                        │
                                          post-processing ─► figures
```

**How the notebook runs things.** The serial steps run in this kernel. The four ensemble stages
(spin-up, open loop, DA and the two collections) run under MPI — one process per ensemble member
plus one coordinator — and are launched from here as subprocesses through `run_stage.py`; their
terminal output is streamed into the cell and the per-rank logs land in `parallel_logs/<stage>/`.
Two of these stages (spin-up, open loop) take hours; every launching cell is guarded by a flag so
you can re-open the notebook later and skip what is already done.

**Prerequisites.** The `pyglda_v2` conda environment (`installation/environment.yml`), MPI
(`mpiexec` on the PATH), the external data tree (`PyGLDA_v2_external_data`) and the WaterGAP
input data. The last step needs `pygmt`; if your kernel has no GMT, the matplotlib previews in
the notebook show the same content.

## 0. Set-up: paths, case definition, helpers

All run settings live on the `RDA` class (`src_FlowControl/Regional_DA.py`). Below they are written
once into `case.json`, which the MPI launcher reads so that every stage sees the same settings.

In [ ]:
import sys, os, json, subprocess, time
from pathlib import Path
import numpy as np
import h5py
import matplotlib.pyplot as plt

ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'src_demo' else Path.cwd().resolve()
assert (ROOT / 'src_FlowControl').is_dir(), f"run this notebook from PyGLDA_v2/src_demo (found {ROOT})"
sys.path.insert(0, str(ROOT))
DEMO = ROOT / 'src_demo'

# WaterGAP's CLI parser reads sys.argv when its controller is imported: give it what it expects
sys.argv = ['notebook', str(DEMO / 'Config_ReWaterGAP.json')]

from src_FlowControl.Regional_DA import RDA
from src_DA.EnumDA import Stage

CASE = {
    'setting_dir':        '/media/user/My Book/Fan/PyGLDA_v2/settings/demo_2',
    'external_data_path': '/media/user/My Book/Fan/PyGLDA_v2_external_data',
    'case':               'demo_2',
    'basin':              'Danube',
    'shp_path':           '/media/user/My Book/Fan/PyGLDA_v2_external_data/Basin/shp/Danube/Danube3subbasins_subbasins.shp',
    'ens':                4,                      # ensemble members (+1 coordinator process under MPI)
    'spin_up_start':      '2000-01-01',
    'spin_up_end':        '2001-12-31',
    'spin_up_years':      5,
    'sim_begin_time':     '2002-01-01',
    'sim_end_time':       '2005-04-30',
    'map_style':          'smooth',               # 'smooth' | 'pixel' for the 2-D harmonic maps
}
for k, v in CASE.items():
    setattr(RDA, k, v)
CASE_JSON = DEMO / 'case.json'
CASE_JSON.write_text(json.dumps(CASE, indent=4))

EXT = Path(CASE['external_data_path'])
RES = EXT / 'Res' / CASE['case']
print('codebase :', ROOT)
print('settings :', CASE['setting_dir'])
print('results  :', RES)

In [ ]:
def run_mpi(stage, n_proc=CASE['ens'] + 1):
    '''Launch one MPI stage through run_stage.py and stream its output into the cell.
    Ranks whose output is redirected write to parallel_logs/<stage>/rank_N.log.'''
    cmd = ['mpiexec', '-n', str(n_proc), 'python', '-u', str(DEMO / 'run_stage.py'), stage,
           str(CASE_JSON), str(DEMO / 'Config_ReWaterGAP.json')]
    print('$', ' '.join(f'"{c}"' if ' ' in c else c for c in cmd), '\n')
    t0 = time.time()
    with subprocess.Popen(cmd, cwd=str(DEMO), stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                          text=True, bufsize=1) as p:
        for line in p.stdout:
            print(line, end='')
    print(f'\n[{stage}] finished with exit code {p.returncode} after {(time.time()-t0)/60:.1f} min')
    if p.returncode != 0:
        print('see parallel_logs/ for the rank logs')
    return p.returncode

def show_log(stage_dir, rank, head=0, tail=15):
    '''print the head and/or tail of a rank log'''
    fn = ROOT / 'parallel_logs' / stage_dir / f'rank_{rank}.log'
    if not fn.exists():
        print('no log:', fn); return
    lines = fn.read_text(errors='replace').replace('\r', '\n').splitlines()
    print(f'--- {fn.relative_to(ROOT)}  ({len(lines)} lines) ---')
    if head: print('\n'.join(lines[:head]))
    if head and tail: print('    ...')
    if tail: print('\n'.join(lines[-tail:]))

# flags: set to False to skip a stage that has already been run
RUN = {'spin_up': True, 'OL': True, 'collect_OL': True, 'DA': True, 'collect_DA': True}

## 1. Configuration

Three JSON files in the settings directory drive an experiment:

* `Config_ReWaterGAP.json` — WaterGAP's own configuration (input/output paths, simulation period,
  which storages are saved);
* `perturbation.json` — how the ensemble is generated (which forcing fields and parameters are
  perturbed, and by how much);
* `DA_setting.json` — the assimilation itself: basin, ensemble size, observation product, which
  storage compartments receive the update, the filter.

`config_external_data()` rewrites every path in the three files to point into your external data
tree, and `read_config_and_save()` writes the case, basin, shapefile, mask and ensemble size from
`RDA` into `DA_setting.json`.

In [ ]:
RDA.config_external_data()
RDA.read_config_and_save()

cfg = json.load(open(Path(CASE['setting_dir']) / 'DA_setting.json'))
print(json.dumps(cfg['basic'], indent=2))
print('\nstorages updated by the filter:', [k for k, v in cfg['model']['layer'].items() if v])
print('observation product          :', cfg['obs']['GRACE']['kind'])

## 2. Basin mask

The basin shapefile is rasterised twice: on WaterGAP's 0.5° grid (used to crop the model output
and to define the observation operator — one area-weighted mean per sub-basin) and on a 1° grid
(used for the GRACE error covariance). The masks are stored as `Basin/mask/<basin>/<basin>_res_*.h5`,
with one dataset per sub-basin plus the whole basin.

In [ ]:
RDA.config_basin_mask()

with h5py.File(EXT / 'Basin/mask' / CASE['basin'] / f"{CASE['basin']}_res_0.5.h5", 'r') as f:
    keys = sorted(k for k in f.keys() if k.startswith('sub_basin'))
    basin = f['basin'][:].astype(bool)
    sub = np.zeros(basin.shape);
    for i, k in enumerate(keys, 1):
        sub[f[k][:].astype(bool)] = i
rows, cols = np.where(basin)
r0, r1, c0, c1 = rows.min(), rows.max() + 1, cols.min(), cols.max() + 1
lat = 89.75 - 0.5 * np.arange(360); lon = -179.75 + 0.5 * np.arange(720)
plt.figure(figsize=(7, 3))
plt.pcolormesh(lon[c0:c1], lat[r0:r1], np.where(basin, sub, np.nan)[r0:r1, c0:c1], cmap='tab10', vmin=0.5, vmax=10.5)
plt.colorbar(label='sub-basin id'); plt.title(f"{CASE['basin']}: {len(keys)} sub-basins, {basin.sum()} cells at 0.5°")
plt.xlabel('lon'); plt.ylabel('lat'); plt.show()

## 3. Ensemble perturbation

An ensemble Kalman filter needs an ensemble whose spread represents the model uncertainty. PyGLDA
builds it by perturbing the climate forcing (precipitation, temperature, radiation) with spatially
and temporally correlated noise, and the model parameters, member by member, for the whole period
from the start of the spin-up to the end of the simulation. The perturbed fields are written under
`Ensemble_input/Ens_<k>/`; member 0 is always the **unperturbed** run, which serves as the
open-loop reference.

In [ ]:
RDA.model_perturbation()

pert = json.load(open(Path(CASE['setting_dir']) / 'perturbation.json'))
print('perturbation settings:', list(pert.keys()))
ens_in = EXT / 'Ensemble_input'
print('ensemble input folders:', sorted(p.name for p in ens_in.glob('Ens_*')))

## 4. GRACE observations

The mascon product (CSR RL06.3) is converted into the observations the filter uses: the monthly
**basin/sub-basin mean TWS** (area-weighted over the 0.5° cells of each sub-basin), the gridded
TWS over the basin box (for the 2-D maps later), and the observation-error covariance from the
DDK3 error samples. `is_diagonal=False` keeps the covariance between sub-basins.

In [ ]:
RDA.get_GRACE_obs(is_diagonal=False)

with h5py.File(Path(cfg['obs']['GRACE']['preprocess_res']) / f"{CASE['basin']}_signal.hdf5", 'r') as f:
    epochs = [t if len(t.split('-')) == 3 else t + '-15' for t in f['time_epoch'][:].astype(str)]
    t = np.array([int(e[:4]) + (int(e[5:7]) - 0.5) / 12 for e in epochs])
    series = {k: f[k][:] for k in f.keys() if k.startswith('sub_basin')}
plt.figure(figsize=(9, 3))
for k, v in series.items():
    plt.plot(t, v, 'o-', ms=3, label=k)
plt.ylabel('GRACE TWS anomaly [mm]'); plt.legend(ncol=3); plt.title('GRACE sub-basin means (whole record prepared)')
plt.axvspan(2002 + 0/12, 2005 + 4/12, color='0.9', label='assimilation period'); plt.show()

## 5. Spin-up (MPI)

Each ensemble member runs WaterGAP over the spin-up period (`spin_up_start`–`spin_up_end`,
repeated `spin_up_years` times) with its own perturbed forcing and parameters, so that the storages
— above all groundwater — reach a state consistent with that member. The final states are written to
`Ensemble_Initialization/Ens_<k>/` and are the starting point of the open loop and of the DA.

Under `mpiexec -n 5`, rank 0 owns the terminal and ranks 1–4 write to
`parallel_logs/spin_up/rank_<k>.log`. **This stage takes hours.**

In [ ]:
if RUN['spin_up']:
    run_mpi('spin_up')
show_log('spin_up', 1, head=0, tail=8)

## 6. Open loop (MPI) and collection

The open loop (OL) is the ensemble run **without** assimilation over the simulation period. It is the
baseline every DA result is compared with, and its ensemble spread is what the filter will later use
as the background error. Output: daily global fields per member under `OL_output/Ens_<k>/`.

The collection step then, for every member, crops the daily global files to the basin box, merges
them into yearly files (`Res/<case>/OL/Ens_<k>/daily_output_<year>.nc`, NaN outside the basin)
and computes the basin and sub-basin mean time series of every storage (`basin_ts_OL.h5`).

In [ ]:
if RUN['OL']:
    run_mpi('OL')
if RUN['collect_OL']:
    run_mpi('collect_OL')

In [ ]:
def basin_ts(stage, variable='tws', basin='basin'):
    '''basin-mean time series of one storage for every ensemble member, from basin_ts_<stage>.h5'''
    out = {}
    for k in range(CASE['ens'] + 1):
        fn = RES / stage / f'Ens_{k}' / f'basin_ts_{stage}.h5'
        if fn.exists():
            with h5py.File(fn, 'r') as f:
                out[k] = f[variable][basin][:]
    return out

ol = basin_ts('OL')
days = np.arange(len(next(iter(ol.values()))))
t_days = 2002 + days / 365.25
plt.figure(figsize=(10, 3.2))
for k, v in ol.items():
    plt.plot(t_days, v, color='0.6' if k else 'tab:blue', lw=0.8 if k else 1.5, label='members' if k == 1 else ('unperturbed (Ens_0)' if k == 0 else None))
plt.plot(t_days, np.mean([v for k, v in ol.items() if k > 0], axis=0), 'k', lw=1.2, label='ensemble mean')
plt.ylabel('TWS [mm]'); plt.title(f"Open loop, {CASE['basin']} basin mean"); plt.legend(); plt.show()

## 7. Data assimilation (MPI) and collection

Now the ensemble is run again over the same period, but at the end of every GRACE month the filter
updates it. The cycle for one month:

1. every member integrates WaterGAP day by day and accumulates the **monthly mean** of the
   assimilated storages (groundwater, soil moisture, snow) over the basin cells;
2. at the end of the month the members' monthly-mean states are gathered on the coordinating rank,
   mapped to sub-basin means with the observation operator **H**, and compared with the GRACE
   observation; the Kalman gain **K = P Hᵀ (H P Hᵀ + R)⁻¹** weights the innovation by the ratio of
   ensemble spread to observation error;
3. the analysis increment of the monthly mean is added to **every day of that month** and to the
   state the next month starts from; storages that are not assimilated (rivers, lakes, wetlands,
   reservoirs, canopy) are carried through unchanged;
4. member 0 (unperturbed) is never updated — it stays the open-loop reference.

Rank 1 owns the terminal in this stage; the banner and the monthly progress of the others are in
`parallel_logs/DA/rank_<k>.log`. Output: daily fields under `DA_output/<case>/Ens_<k>/`, then the
same collection as for the OL into `Res/<case>/DA/`.

In [ ]:
if RUN['DA']:
    run_mpi('DA')
if RUN['collect_DA']:
    run_mpi('collect_DA')
show_log('DA', 2, head=28, tail=6)

## 8. Post-processing: OL vs DA vs GRACE

`post_processing()` gathers the basin time series of all members for both stages
(`Res_OL.h5`, `Res_DA.h5`), the GRACE basin series (`GRACE_<basin>.h5`), and runs the per-grid-cell
harmonic analysis (trend, annual and semi-annual cycle) on the OL, DA and GRACE fields
(`Harmonic_{OL,DA,GRACE}.nc`, see `src_auxiliary/ts.py`).

The plot below is the essential result: does the DA ensemble mean follow the GRACE observations
better than the open loop? Remember what is being compared — the DA reproduces the assimilated
quantity (sub-basin monthly means) largely by construction; the daily shape inside a month is the
model's, and the amount of correction is governed by the ensemble spread relative to the GRACE
error.

In [ ]:
RDA.post_processing()

from src_postprocessing.statistical_analysis import BasinAverageAnalysis_post
bp = BasinAverageAnalysis_post(ens=CASE['ens'], case=CASE['case'], basin=CASE['basin'],
                               date_begin=CASE['sim_begin_time'], date_end=CASE['sim_end_time'])
S_OL = bp.load_states(load_dir=RES, prefix='OL')
S_DA = bp.load_states(load_dir=RES, prefix='DA')
G = bp.load_GRACE(load_dir=RES, prefix=CASE['basin'])

basins = [k for k in S_OL.keys() if k != 'time']
fig, axes = plt.subplots(len(basins), 1, figsize=(10, 2.6 * len(basins)), sharex=True)
for ax, b in zip(np.atleast_1d(axes), basins):
    ol_mean = np.mean([S_OL[b]['tws'][k] for k in range(1, CASE['ens'] + 1)], axis=0)
    da_mean = np.mean([S_DA[b]['tws'][k] for k in range(1, CASE['ens'] + 1)], axis=0)
    gb = 'basin_0' if b == 'basin' else 'basin_' + b.split('_')[-1]
    ax.plot(S_OL['time'], ol_mean, 'b--', lw=1, label='OL')
    ax.plot(S_DA['time'], da_mean, 'g-', lw=1, label='DA')
    ax.plot(G['time'], G['original'][gb], 'ko', ms=4, label='GRACE')
    ax.set_ylabel('TWS [mm]'); ax.set_title(f"{CASE['basin']} {b}"); ax.grid(alpha=.3)
axes[0].legend(ncol=3) if len(basins) > 1 else plt.legend(ncol=3)
plt.tight_layout(); plt.show()

## 9. Where in the basin did the DA act? — harmonic maps

The per-cell fits give maps of the linear trend, the annual amplitude and the day of year of the
annual maximum, for OL, DA and GRACE. A quick matplotlib preview first (values straight from the
netCDF files); the publication-style pygmt figure follows in the next cell.

In [ ]:
import xarray as xr
from src_postprocessing.statistical_analysis import HarmonicMapAnalysis

maps = {s: HarmonicMapAnalysis.load(EXT / 'Res', CASE['case'], s) for s in (Stage.OL, Stage.DA, 'GRACE')}
rows = [('trend', 'trend [mm/yr]', 'RdBu_r', True), ('annual_amp', 'annual amplitude [mm]', 'viridis', False),
        ('annual_peak_doy', 'annual peak [day of year]', 'twilight', False)]
fig, axes = plt.subplots(len(rows), 3, figsize=(13, 3.2 * len(rows)))
for r, (q, label, cmap, sym) in enumerate(rows):
    grids = []
    for s in maps:
        ds = maps[s]; name = f'tws_{q}_ensmean' if f'tws_{q}_ensmean' in ds else f'tws_{q}'
        grids.append(ds[name])
    vals = np.concatenate([g.values.ravel() for g in grids]); vals = vals[np.isfinite(vals)]
    lo, hi = np.percentile(vals, [2, 98]);
    if sym: m = max(abs(lo), abs(hi)); lo, hi = -m, m
    for c, (s, g) in enumerate(zip(maps, grids)):
        im = axes[r, c].pcolormesh(g['lon'], g['lat'], g.values, cmap=cmap, vmin=lo, vmax=hi, shading='auto')
        axes[r, c].set_title(f"{getattr(s, 'name', s)}: {label}", fontsize=10)
    plt.colorbar(im, ax=axes[r, :], shrink=0.8, pad=0.01)
plt.show()

In [ ]:
# publication-style pygmt figures (needs pygmt/GMT in this kernel): basin ensembles,
# OL/DA/GRACE per sub-basin, and the 2-D harmonic maps (style = CASE['map_style'])
RDA.visualization()

## 10. Reading the result, and what to try next

* **Does the filter work?** Mechanically yes if the DA mean follows GRACE month by month with no
  drift or instability, and the increments are physically plausible. Whether it *adds information*
  is a separate question that the assimilated quantity cannot answer by itself — validate against
  something not assimilated (withheld GRACE months, discharge, groundwater wells), or look at the
  spatial pattern within the sub-basins in the maps above.
* **Ensemble size.** Four members are enough to exercise the machinery, not to estimate the
  covariances the filter relies on. Twenty to thirty is the range for a real experiment; the
  memory-light collection step makes that affordable.
* **How much is the model trusted?** The gain in observation space, H P Hᵀ / (H P Hᵀ + R), tells
  you. If it is small the filter stays with the model because the spread is small relative to the
  GRACE error — more members, stronger perturbations or inflation move it.
* **Towards the global case.** For gridded GRACE (3° cells) the same pipeline applies with one
  `sub_basin_k` per GRACE cell in the mask; the local ensemble transform Kalman filter in
  `src_DA/LETKF.py` is the filter for that setting.

To re-run a single stage later, set the corresponding entry of `RUN` and execute its cell, or call
`run_stage.py` from a terminal:

```
cd src_demo
mpiexec -n 5 python -u run_stage.py collect_DA case.json Config_ReWaterGAP.json
```